# Train the license plate detectors

Trains two YOLO11 models for the in-browser app:

| Variant | Model | Use |
| :-- | :-- | :-- |
| `fast` | YOLO11n | live webcam, small download |
| `accurate` | YOLO11s | photos, best accuracy |

**How to run**
1. **Runtime → Change runtime type →** pick the strongest GPU you have (A100, then L4, then T4).
2. **Runtime → Run all.** Keep this tab open: on an A100 the whole run takes about 1.5–2.5 hours.
3. The last cell downloads `plates-model.zip`. Unzip it into the repo root.

**Data:** Vehicle Registration Plates (Augmented Startups, Roboflow, CC BY 4.0) plus Open Images V7
"Vehicle registration plate" (CC BY 4.0 annotations), which adds varied street and phone photos.

In [ ]:
# Settings. Lower EPOCHS or OPEN_IMAGES for a quicker run.
VARIANTS = ["fast", "accurate"]
IMGSZ = 768          # plates are small; above 640 helps recall
EPOCHS = 120         # early stopping ends sooner if validation stops improving
OPEN_IMAGES = 6000   # max extra training images from Open Images (0 = off)

In [ ]:
# Which GPU did we get?
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone -q https://github.com/Abiks06/license-plate-detection.git
%cd license-plate-detection
!pip install -q -r training/requirements.txt fiftyone

In [ ]:
!python training/scripts/prepare_dataset.py --open-images {OPEN_IMAGES}

In [ ]:
for v in VARIANTS:
    !python training/scripts/train.py --variant {v} --device 0 --imgsz {IMGSZ} --epochs {EPOCHS} --workers 4

In [ ]:
for v in VARIANTS:
    !python training/scripts/export.py --variant {v} --device 0
!cat web/public/model/models.json

In [ ]:
# Bundle browser models, weights, metrics and training plots, then download.
import shutil, pathlib
bundle = pathlib.Path("bundle")
shutil.rmtree(bundle, ignore_errors=True)
shutil.copytree("web/public/model", bundle / "web/public/model")
for v in VARIANTS:
    run, dest = pathlib.Path("runs") / v, bundle / "training/results" / v
    dest.mkdir(parents=True)
    shutil.copy(run / "weights/best.pt", dest / "best.pt")
    for f in ["results.csv", "results.png", "confusion_matrix.png", "BoxPR_curve.png", "BoxF1_curve.png", "val_batch0_pred.jpg", "args.yaml"]:
        if (run / f).exists():
            shutil.copy(run / f, dest / f)
shutil.make_archive("plates-model", "zip", bundle)
from google.colab import files
files.download("plates-model.zip")